In [1]:
import duckdb
import polars as pl
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score
)

In [2]:
con = duckdb.connect()

In [3]:
train_sample = con.execute("""

SELECT *

FROM read_parquet(
    '../data/modeling/features.parquet'
)

WHERE fecha_hora < '2026-05-01'

USING SAMPLE 10%

""").pl()

In [4]:
print(train_sample.shape)

(3331606, 49)


In [5]:
test_sample = con.execute("""

SELECT *

FROM read_parquet(
    '../data/modeling/features.parquet'
)

WHERE fecha_hora >= '2026-05-01'

USING SAMPLE 15%

""").pl()

In [6]:
print(test_sample.shape)

(1068457, 49)


In [7]:
X_train = train_sample.drop([
    "target",
    "fecha_hora"
])

y_train = train_sample["target"]


X_test = test_sample.drop([
    "target",
    "fecha_hora"
])

y_test = test_sample["target"]

In [8]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (3331606, 47)
y_train: (3331606,)
X_test: (1068457, 47)
y_test: (1068457,)


## Baseline de persistencia

Antes de evaluar el modelo, medimos qué tan bien predice la **persistencia pura**: usar el valor de la hora anterior (`lag_1`) y el de la misma hora de la semana anterior (`lag_168`) como predicción directa. Todo modelo debe superar este piso para justificar su existencia.

In [9]:
# Baseline 1: persistencia lag_1 (valor de la hora anterior)
pred_lag1 = test_sample["intensidad_media_lag_1"].to_numpy()

# Baseline 2: persistencia semanal lag_168 (misma hora, semana anterior)
pred_lag168 = test_sample["intensidad_media_lag_168"].to_numpy()

y_true = y_test.to_numpy()

for nombre, pred_bl in [("lag_1", pred_lag1), ("lag_168", pred_lag168)]:
    print(f"BASELINE {nombre}")
    print(f"  MAE : {mean_absolute_error(y_true, pred_bl):.4f}")
    print(f"  RMSE: {root_mean_squared_error(y_true, pred_bl):.4f}")
    print(f"  R²  : {r2_score(y_true, pred_bl):.4f}")
    print()

BASELINE lag_1
  MAE : 360.0836
  RMSE: 696.1476
  R²  : -5146.4794

BASELINE lag_168
  MAE : 360.2455
  RMSE: 694.7541
  R²  : -5125.8919



In [10]:
X_train.schema

Schema([('id_sensor', Int32),
        ('hora', Int32),
        ('intensidad_media', Float64),
        ('intensidad_max', Float64),
        ('intensidad_min', Float64),
        ('ocupacion_media', Float64),
        ('ocupacion_max', Float64),
        ('velocidad_media', Float64),
        ('velocidad_min', Float64),
        ('num_mediciones', Int64),
        ('num_error_E', Float64),
        ('porcentaje_calidad', Float64),
        ('año', Int64),
        ('mes', Int64),
        ('trimestre', Int64),
        ('dia', Int64),
        ('dia_semana', Int64),
        ('fin_semana', Boolean),
        ('tipo_elem', String),
        ('distrito', Int32),
        ('latitud', Float64),
        ('longitud', Float64),
        ('hora_sin', Float64),
        ('hora_cos', Float64),
        ('dia_semana_sin', Float64),
        ('dia_semana_cos', Float64),
        ('mes_sin', Float64),
        ('mes_cos', Float64),
        ('intensidad_media_lag_1', Float64),
        ('intensidad_media_lag_24', Float64),


In [11]:
X_train = X_train.with_columns(
    pl.col("tipo_elem").cast(pl.Categorical)
)

X_test = X_test.with_columns(
    pl.col("tipo_elem").cast(pl.Categorical)
)

In [12]:
X_train = X_train.with_columns(
    pl.col(pl.Float64).cast(pl.Float32),
    pl.col(pl.Int64).cast(pl.Int32)
)

X_test = X_test.with_columns(
    pl.col(pl.Float64).cast(pl.Float32),
    pl.col(pl.Int64).cast(pl.Int32)
)

In [13]:
X_train = X_train.with_columns(
    pl.col("fin_semana").cast(pl.Int8)
)

X_test = X_test.with_columns(
    pl.col("fin_semana").cast(pl.Int8)
)

In [14]:
X_train.schema

Schema([('id_sensor', Int32),
        ('hora', Int32),
        ('intensidad_media', Float32),
        ('intensidad_max', Float32),
        ('intensidad_min', Float32),
        ('ocupacion_media', Float32),
        ('ocupacion_max', Float32),
        ('velocidad_media', Float32),
        ('velocidad_min', Float32),
        ('num_mediciones', Int32),
        ('num_error_E', Float32),
        ('porcentaje_calidad', Float32),
        ('año', Int32),
        ('mes', Int32),
        ('trimestre', Int32),
        ('dia', Int32),
        ('dia_semana', Int32),
        ('fin_semana', Int8),
        ('tipo_elem', Categorical),
        ('distrito', Int32),
        ('latitud', Float32),
        ('longitud', Float32),
        ('hora_sin', Float32),
        ('hora_cos', Float32),
        ('dia_semana_sin', Float32),
        ('dia_semana_cos', Float32),
        ('mes_sin', Float32),
        ('mes_cos', Float32),
        ('intensidad_media_lag_1', Float32),
        ('intensidad_media_lag_24', Float32)

In [15]:
X_train_pd = X_train.to_pandas()
X_test_pd = X_test.to_pandas()

In [16]:
X_train_pd["tipo_elem"] = X_train_pd["tipo_elem"].astype("category")
X_test_pd["tipo_elem"] = X_test_pd["tipo_elem"].astype("category")

In [17]:
categorical_features = [
    X_train_pd.columns.get_loc("tipo_elem")
]

In [18]:
print(X_train_pd.shape)
print(X_test_pd.shape)

(3331606, 47)
(1068457, 47)


In [19]:
X_train_pd.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 3331606 entries, 0 to 3331605
Data columns (total 47 columns):
 #   Column                      Dtype   
---  ------                      -----   
 0   id_sensor                   int32   
 1   hora                        int32   
 2   intensidad_media            float32 
 3   intensidad_max              float32 
 4   intensidad_min              float32 
 5   ocupacion_media             float32 
 6   ocupacion_max               float32 
 7   velocidad_media             float32 
 8   velocidad_min               float32 
 9   num_mediciones              int32   
 10  num_error_E                 float32 
 11  porcentaje_calidad          float32 
 12  año                         int32   
 13  mes                         int32   
 14  trimestre                   int32   
 15  dia                         int32   
 16  dia_semana                  int32   
 17  fin_semana                  int8    
 18  tipo_elem                   category
 19  distrito   

In [20]:
modelo = HistGradientBoostingRegressor(
    max_iter=100,
    learning_rate=0.1,
    max_leaf_nodes=31,
    categorical_features=categorical_features,
    random_state=42
)

In [21]:
modelo.fit(
    X_train_pd,
    y_train.to_numpy()
)

,"categorical_features categorical_features: array-like of {bool, int, str} of shape (n_features) or shape (n_categorical_features,), default='from_dtype'Indicates the categorical features.- None : no feature will be considered categorical.- boolean array-like : boolean mask indicating categorical features.- integer array-like : integer indices indicating categorical features.- str array-like: names of categorical features (assuming the training data has feature names).- `""from_dtype""`: dataframe columns with dtype ""Categorical"" and ""Enum"" are considered to be categorical features. The input must be a dataframe that is supported by narwhals (or supports it): :func:`narwhals.from_native` must work. This is the case, for instance, for pandas and polars DataFrames.For each categorical feature, there must be at most `max_bins` uniquecategories. Negative values for categorical features encoded as numericdtypes are treated as missing values. All categorical values areconverted to floating point numbers. This means that categorical valuesof 1.0 and 1 are treated as the same category.Read more in the :ref:`User Guide <categorical_support_gbdt>` and:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_categorical.py`... versionadded:: 0.24.. versionchanged:: 1.2 Added support for feature names... versionchanged:: 1.4 Added `""from_dtype""` option... versionchanged:: 1.6 The default value changed from `None` to `""from_dtype""`.",[18]
,"random_state random_state: int, RandomState instance or None, default=NonePseudo-random number generator to control the subsampling in thebinning process, and the train/validation data split if early stoppingis enabled.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"loss loss: {'squared_error', 'absolute_error', 'gamma', 'poisson', 'quantile'}, default='squared_error'The loss function to use in the boosting process. Note that the""squared error"", ""gamma"" and ""poisson"" losses actually implement""half least squares loss"", ""half gamma deviance"" and ""half poissondeviance"" to simplify the computation of the gradient. Furthermore,""gamma"" and ""poisson"" losses internally use a log-link, ""gamma""requires ``y > 0`` and ""poisson"" requires ``y >= 0``.""quantile"" uses the pinball loss... versionchanged:: 0.23 Added option 'poisson'... versionchanged:: 1.1 Added option 'quantile'... versionchanged:: 1.3 Added option 'gamma'.",'squared_error'
,"quantile quantile: float, default=NoneIf loss is ""quantile"", this parameter specifies which quantile to be estimatedand must be between 0 and 1.",None
,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.1
,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees.",100
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",31
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",None
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",20
,"l2_regularization l2_regularization: float, default=0The L2 regularization parameter penalizing leaves with small hessians.Use ``0`` for no regularization (default).",0.0
,"max_features max_features: float, default=1.0Proportion of randomly chosen features in each and every node split.This is a form of regularization, smaller values make the trees weakerlearners and might prevent overfitting.If interaction constraints fr

In [22]:
pred = modelo.predict(X_test_pd)

mae = mean_absolute_error(
    y_test,
    pred
)

rmse = root_mean_squared_error(
    y_test,
    pred
)

r2 = r2_score(
    y_test,
    pred
)

print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²: {r2:.4f}")

MAE: 1.6131
RMSE: 3.6616
R²: 0.8576


In [23]:
pred_train = modelo.predict(X_train_pd)

print("TRAIN")
print("MAE :", mean_absolute_error(y_train, pred_train))
print("RMSE:", root_mean_squared_error(y_train, pred_train))
print("R²  :", r2_score(y_train, pred_train))

TRAIN
MAE : 1.3402910614148202
RMSE: 3.0499326790927954
R²  : 0.8690969428747298


## Modelo V2: más capacidad + early stopping

Train ≈ Test en el modelo base indica subajuste. Aumentamos `max_iter` y `max_leaf_nodes`, con `early_stopping` sobre un 10% de validación para frenar automáticamente cuando deja de mejorar (evita overfitting sin tunear a mano el número de iteraciones).

In [24]:
modelo_v2 = HistGradientBoostingRegressor(
    max_iter=1000,
    learning_rate=0.1,
    max_leaf_nodes=63,
    categorical_features=categorical_features,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=20,
    random_state=42
)

modelo_v2.fit(
    X_train_pd,
    y_train.to_numpy()
)

print(f"Iteraciones efectivas: {modelo_v2.n_iter_}")

Iteraciones efectivas: 1000


In [25]:
pred_v2 = modelo_v2.predict(X_test_pd)

print("MODELO V2 — TEST")
print(f"  MAE : {mean_absolute_error(y_test, pred_v2):.4f}")
print(f"  RMSE: {root_mean_squared_error(y_test, pred_v2):.4f}")
print(f"  R²  : {r2_score(y_test, pred_v2):.4f}")

pred_v2_train = modelo_v2.predict(X_train_pd)
print("MODELO V2 — TRAIN")
print(f"  MAE : {mean_absolute_error(y_train, pred_v2_train):.4f}")
print(f"  RMSE: {root_mean_squared_error(y_train, pred_v2_train):.4f}")
print(f"  R²  : {r2_score(y_train, pred_v2_train):.4f}")

MODELO V2 — TEST
  MAE : 1.5365
  RMSE: 3.5791
  R²  : 0.8639
MODELO V2 — TRAIN
  MAE : 1.1401
  RMSE: 2.4422
  R²  : 0.9161


## Comparativa final

| Modelo | MAE | RMSE | R² |
|---|---|---|---|
| Baseline lag_1 | _completar_ | _completar_ | _completar_ |
| Baseline lag_168 | _completar_ | _completar_ | _completar_ |
| HGB base (100 iter, 31 hojas) | 1.6835 | 3.7992 | 0.8758 |
| HGB V2 (early stopping, 63 hojas) | _completar_ | _completar_ | _completar_ |